In [1]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

image_transform = A.Compose([
    A.Resize(512, 512),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

mask_transform = A.Compose([
    A.Resize(512, 512),
    ToTensorV2()
])

In [2]:
from torch.utils.data import Dataset
import torch
import cv2
import numpy as np
from pathlib import Path

class SegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_dir, image_transform=None, mask_transform=None):
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.image_paths = sorted(list(self.image_dir.glob("*.jpg")))
        self.image_transform = image_transform
        self.mask_transform = mask_transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        mask_path = self.mask_dir / image_path.with_suffix(".tif").name

        image = cv2.imread(str(image_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

        # Apply transforms
        if self.mask_transform:
            mask = self.mask_transform(image=mask)["image"]
        if self.image_transform:
            image = self.image_transform(image=image)["image"]

        # Make sure mask is a LongTensor for CrossEntropyLoss
        mask = torch.from_numpy(mask).long()

        return image, mask


In [3]:
import os
from pathlib import Path
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset

class MultiMaskSegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_base_dir, lesion_folders, image_transform=None, mask_transform=None):
        self.image_dir = Path(image_dir)
        self.mask_base_dir = Path(mask_base_dir)
        self.lesion_folders = lesion_folders
        self.image_transform = image_transform
        self.mask_transform = mask_transform

        self.image_paths = sorted(list(self.image_dir.glob("*.jpg")))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        base_name = img_path.stem  # filename without extension

        masks = []
        for lesion in self.lesion_folders:
            mask_path = self.mask_base_dir / lesion / f"{base_name}.tif"
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            if mask is None:
                # Create empty mask if not found
                mask = np.zeros((image.shape[0], image.shape[1]), dtype=np.uint8)
            masks.append(mask)

        # Stack masks along channel dimension: shape (H, W, num_lesions)
        combined_mask = np.stack(masks, axis=-1)

        # Apply transforms if provided
        if self.image_transform:
            augmented = self.image_transform(image=image, mask=combined_mask)
            image = augmented['image']
            combined_mask = augmented['mask']

        if self.mask_transform:
            combined_mask = self.mask_transform(combined_mask)

        return image, combined_mask


image_dir = "/Users/poojasrisundaresan/Desktop/Internship/A. Segmentation/1. Original Images/a. Training Set"
mask_base_dir = "/Users/poojasrisundaresan/Desktop/Internship/A. Segmentation/2. All Segmentation Groundtruths/a. Training Set"

lesion_folders = [
    "1. Microaneurysms",
    "2. Hemorrhages",
    "3. Hard Exudates",
    "4. Soft Exudates",
    "5. Optic Disc"
]

dataset = MultiMaskSegmentationDataset(
    image_dir=image_dir,
    mask_base_dir=mask_base_dir,
    lesion_folders=lesion_folders,
    image_transform=image_transform, 
    mask_transform=mask_transform
)

from torch.utils.data import DataLoader
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# Test one batch
images, masks = next(iter(dataloader))
print("Image shape:", images.shape) 
print("Mask shape:", masks.shape)  

[ WARN:0@1.081] global loadsave.cpp:275 findDecoder imread_('/Users/poojasrisundaresan/Desktop/Internship/A. Segmentation/2. All Segmentation Groundtruths/a. Training Set/1. Microaneurysms/IDRiD_01.tif'): can't open/read file: check file path/integrity
[ WARN:0@1.081] global loadsave.cpp:275 findDecoder imread_('/Users/poojasrisundaresan/Desktop/Internship/A. Segmentation/2. All Segmentation Groundtruths/a. Training Set/2. Hemorrhages/IDRiD_01.tif'): can't open/read file: check file path/integrity
[ WARN:0@1.081] global loadsave.cpp:275 findDecoder imread_('/Users/poojasrisundaresan/Desktop/Internship/A. Segmentation/2. All Segmentation Groundtruths/a. Training Set/3. Hard Exudates/IDRiD_01.tif'): can't open/read file: check file path/integrity
[ WARN:0@1.081] global loadsave.cpp:275 findDecoder imread_('/Users/poojasrisundaresan/Desktop/Internship/A. Segmentation/2. All Segmentation Groundtruths/a. Training Set/4. Soft Exudates/IDRiD_01.tif'): can't open/read file: check file path/int

KeyError: 'You have to pass data to augmentations as named arguments, for example: aug(image=image)'

In [ ]:
from pathlib import Path

image_dir_path = Path(image_dir)
mask_dir_path = Path(mask_dir)

print("Number of images:", len(list(image_dir_path.glob("*.jpg"))))
print("Number of masks:", len(list(mask_dir_path.glob("*.tif"))))

Number of images: 54
Number of masks: 0
